# cudf-bench: GPU-vs-CPU transient split (Step 3d)

**Before running:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`. ~6 min. **Click Allow on the download at the end.**

Known so far: skewed groupby runs ~1.6x slow for the first ~5 calls per process, then snaps to fast. Not the allocator (survives preallocated pool). This run answers three questions at once:

1. **Where does the extra time go?** GPU event timers split each call into GPU busy-time vs CPU-side time.
2. **Is it time-based or call-count-based?** One run sleeps 2s between calls — if the transient stretches, some cache decays with time.
3. **Is it groupby-specific?** `sort` runs the same schedule under skew.

In [ ]:
!nvidia-smi -L

In [ ]:
try:
    import cudf
except ImportError:
    %pip install -q cudf-cu12
    import cudf
print("cudf", cudf.__version__)

In [ ]:
%cd /content
![ -d cudf-bench ] || git clone -q https://github.com/alexislowys/cudf-bench.git
%cd /content/cudf-bench
# VM clone is disposable — force it to match GitHub exactly, discard any leftovers
!git fetch -q && git reset -q --hard origin/main && git clean -qfd results
!rm -f results/transient2.csv
%pip install -q -e .

In [ ]:
# 1+2: groupby, skewed vs uniform, back-to-back vs 2s gaps
!python -m benchmark.transient --op groupby --skew 1.5 --str-cols 2 --iters 12 --out results/transient2.csv
!python -m benchmark.transient --op groupby --skew 0   --str-cols 2 --iters 12 --out results/transient2.csv
!python -m benchmark.transient --op groupby --skew 1.5 --str-cols 2 --iters 12 --gap 2 --out results/transient2.csv
# 3: does sort show the same skew transient?
!python -m benchmark.transient --op sort --skew 1.5 --str-cols 2 --iters 12 --out results/transient2.csv
!python -m benchmark.transient --op sort --skew 0   --str-cols 2 --iters 12 --out results/transient2.csv

## Quick look

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

df = pd.read_csv("results/transient2.csv")
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharex=True)
for (op, skew, gap), g in df.groupby(["op", "skew", "gap"]):
    label = f"{op} skew={skew}" + (f" gap={gap}s" if gap else "")
    axes[0].plot(g["iter"], g["wall_ms"], marker="o", label=label)
    axes[1].plot(g["iter"], g["gpu_ms"], marker="o", label=label)
axes[0].set_title("wall time per call"); axes[1].set_title("GPU busy time per call")
for ax in axes: ax.set_xlabel("call #"); ax.set_ylabel("ms"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

## Auto-download (click Allow)

In [ ]:
from google.colab import files
files.download("/content/cudf-bench/results/transient2.csv")